In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
1,model_13_5_24,0.787604,0.179402,-2.612439,-1.001258,-2.301236,0.347801,1.343737,0.755854,0.093070,0.424462,0.901779,0.589747,1.175776,0.614853,108.112250,172.712668,"Hidden Size=[13], regularizer=0.2, learning_ra..."
3,model_7_4_11,0.785600,0.115078,0.381513,0.841239,0.780729,0.351082,1.449068,0.599050,0.949687,0.774369,1.245416,0.592522,1.245029,0.617747,92.093470,146.942882,"Hidden Size=[11], regularizer=0.05, learning_r..."
4,model_7_4_10,0.785589,0.114434,0.388227,0.842305,0.782552,0.351100,1.450122,0.592547,0.943310,0.767929,1.254356,0.592537,1.245041,0.617763,92.093366,146.942778,"Hidden Size=[11], regularizer=0.05, learning_r..."
5,model_7_4_12,0.785573,0.115708,0.374782,0.840154,0.778887,0.351126,1.448036,0.605570,0.956181,0.780875,1.236547,0.592559,1.245059,0.617785,92.093220,146.942632,"Hidden Size=[11], regularizer=0.05, learning_r..."
6,model_7_4_9,0.785539,0.113777,0.394923,0.843352,0.784357,0.351182,1.451199,0.586062,0.937051,0.761556,1.263371,0.592606,1.245099,0.617835,92.092900,146.942312,"Hidden Size=[11], regularizer=0.05, learning_r..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,model_5_3_3,0.522524,-0.469091,0.677292,-34.018669,-1.914988,0.781871,2.405650,0.137864,1.248117,0.692991,2.401356,0.884235,1.545687,0.921878,90.492131,145.341543,"Hidden Size=[11], regularizer=0.2, learning_ra..."
1456,model_11_2_10,0.522404,0.202964,0.334514,-0.027193,0.373510,0.782068,1.305155,0.928627,0.051894,0.490260,1.069978,0.884346,1.458492,0.921994,98.491628,158.216544,"Hidden Size=[12], regularizer=0.05, learning_r..."
1459,model_10_3_21,0.522078,-0.306067,-13.661282,-2.133801,-1.804758,0.782601,2.138697,0.186408,4.660712,2.423560,0.566979,0.884648,1.458805,0.922309,98.490263,158.215179,"Hidden Size=[12], regularizer=0.2, learning_ra..."
1460,model_17_5_14,0.521882,0.130865,-12.222751,-0.107941,-0.003609,0.782922,1.423217,0.312898,1.240529,0.776714,1.817322,0.884829,1.347722,0.922498,114.489443,183.965365,"Hidden Size=[14], regularizer=0.2, learning_ra..."
